In [3]:
!pip install pandas
!pip install sqlglot


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\madal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/526.1 kB ? eta -:--:--
   -- ------------------------------------- 30.7/526.1 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------  522.2/526.1 kB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 526.1/526.1 kB 6.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\madal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


working code

In [13]:
import os
import json
from typing import Dict, List, Any
import sqlglot
from sqlglot import exp


class SQLDependencyParser:
    def __init__(self, sql_folder: str):
        self.sql_folder = sql_folder
        self.dependencies: Dict[str, Any] = {}
        self.table_aliases: Dict[str, Dict[str, str]] = {}

    def parse_all_sql_files(self) -> Dict[str, Any]:
        """Parse all .sql files in the given folder and build the dependency graph."""
        for file_name in os.listdir(self.sql_folder):
            if file_name.endswith(".sql"):
                file_path = os.path.join(self.sql_folder, file_name)
                print(f"🔍 Parsing {file_name}")
                self._parse_sql_file(file_path)
        return self.dependencies

    def _parse_sql_file(self, file_path: str):
        """Parse a single SQL file and extract its relationships."""
        file_key = os.path.splitext(os.path.basename(file_path))[0]
        with open(file_path, "r", encoding="utf-8") as f:
            sql_content = f.read()

        # Parse SQL with Oracle compatibility (handles (+))
        try:
            parsed = sqlglot.parse_one(sql_content, read="oracle")
        except Exception as e:
            print(f"⚠️ Failed to parse {file_key}: {e}")
            return

        self.dependencies[file_key] = {
            "depends": [],
            "left_join": [],
            "right_join": [],
            "inner_join": [],
            "columns": {},
        }

        self._extract_table_aliases(parsed, file_key)
        self._extract_table_references(parsed, file_key)
        self._extract_join_info(parsed, file_key)
        self._extract_implicit_oracle_joins(parsed, file_key)  # NEW
        self._extract_selected_columns(parsed, file_key)

    def _extract_table_aliases(self, expression: exp.Expression, file_key: str):
        aliases = {}
        for table in expression.find_all(exp.Table):
            alias = table.alias_or_name
            name = table.name
            aliases[alias] = name
        self.table_aliases[file_key] = aliases

    def _extract_table_references(self, expression: exp.Expression, file_key: str):
        tables = {t.name for t in expression.find_all(exp.Table)}
        self.dependencies[file_key]["depends"].extend(sorted(tables))

    def _get_join_type_correct(self, join: exp.Join) -> str:
        """Improved: Determine exact JOIN type robustly for multiple dialects"""
        kind = join.args.get("kind")
        if kind:
            kind = str(kind).upper().strip()

        if not kind or kind in {"", "JOIN"}:
            raw_sql = str(join)
            if "LEFT JOIN" in raw_sql.upper():
                kind = "LEFT"
            elif "RIGHT JOIN" in raw_sql.upper():
                kind = "RIGHT"
            elif "FULL JOIN" in raw_sql.upper() or "OUTER JOIN" in raw_sql.upper():
                kind = "FULL"
            else:
                kind = "INNER"

        # Oracle (+) detection
        on_expr = join.args.get("on")
        if on_expr:
            on_str = on_expr.sql(dialect="oracle").upper()
            if "(+)" in on_str and "=" in on_str:
                eq_expr = list(on_expr.find_all(exp.EQ))
                if eq_expr:
                    eq_str = eq_expr[0].sql(dialect="oracle").upper()
                    if "(+)" in eq_str.split("=")[0]:
                        kind = "RIGHT"
                    elif "(+)" in eq_str.split("=")[1]:
                        kind = "LEFT"

        if "LEFT" in kind:
            return "left_join"
        elif "RIGHT" in kind:
            return "right_join"
        elif "FULL" in kind or "OUTER" in kind:
            return "inner_join"
        else:
            return "inner_join"

    def _extract_column_from_expression(self, expr: exp.Expression) -> str:
        """Extract a column name from an expression if possible."""
        if isinstance(expr, exp.Column):
            return expr.name
        return str(expr)

    def _extract_join_info(self, expression: exp.Expression, file_key: str):
        """Extract explicit JOINs from SQL."""
        for join in expression.find_all(exp.Join):
            join_type = self._get_join_type_correct(join)

            on_condition = join.args.get("on")
            if not on_condition:
                continue

            eq_conditions = [c for c in on_condition.find_all(exp.EQ)]
            for eq in eq_conditions:
                left_table = getattr(eq.left, "table", None)
                right_table = getattr(eq.right, "table", None)

                left_name = self.table_aliases[file_key].get(left_table, left_table)
                right_name = self.table_aliases[file_key].get(right_table, right_table)

                tables = [left_name, right_name]
                column = self._extract_column_from_expression(eq.left)

                join_data = {"table": tables, "column": column}
                self.dependencies[file_key][join_type].append(join_data)
                print(f"    {join_type.upper()}: {tables} on {column}")

    def _extract_implicit_oracle_joins(self, expression: exp.Expression, file_key: str):
        """Detect Oracle-style implicit joins using (+) in WHERE clause."""
        for where in expression.find_all(exp.Where):
            for condition in where.find_all(exp.EQ):
                cond_str = condition.sql(dialect="oracle").upper()
                if "(+)" not in cond_str:
                    continue

                left_expr = condition.left
                right_expr = condition.right
                left_table = getattr(left_expr, "table", None)
                right_table = getattr(right_expr, "table", None)

                left_name = self.table_aliases[file_key].get(left_table, left_table)
                right_name = self.table_aliases[file_key].get(right_table, right_table)

                if "(+)" in cond_str.split("=")[0]:
                    join_type = "right_join"
                    tables = [left_name, right_name]
                    column = self._extract_column_from_expression(right_expr)
                else:
                    join_type = "left_join"
                    tables = [left_name, right_name]
                    column = self._extract_column_from_expression(left_expr)

                join_data = {"table": tables, "column": column or "unknown"}
                self.dependencies[file_key][join_type].append(join_data)
                print(f"    {join_type.upper()} (Oracle implicit): {tables} on {column}")

    def _extract_selected_columns(self, expression: exp.Expression, file_key: str):
        """Map selected columns to their source tables."""
        columns_map = {}
        for col in expression.find_all(exp.Column):
            table = getattr(col, "table", None)
            column = col.name
            table_name = self.table_aliases[file_key].get(table, table)
            if table_name:
                columns_map.setdefault(table_name, set()).add(column)

        self.dependencies[file_key]["columns"] = {
            k: sorted(list(v)) for k, v in columns_map.items()
        }

    def export_to_json(self, output_path: str):
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(self.dependencies, f, indent=4)
        print(f"✅ Exported dependency graph to {output_path}")


if __name__ == "__main__":
    folder = "sample1"  # Folder containing .sql files
    parser = SQLDependencyParser(folder)
    result = parser.parse_all_sql_files()
    parser.export_to_json("dependency_graph.json")


Outer join syntax using the (+) operator is not supported.


🔍 Parsing file1.sql
    LEFT_JOIN: ['customers', 'accounts'] on customer_id
🔍 Parsing file2.sql
    INNER_JOIN: ['file1', 'transactions'] on customer_id
    LEFT_JOIN: ['transactions', 'branches'] on branch_id
🔍 Parsing file3.sql
    LEFT_JOIN: ['file1', 'file2'] on customer_id
    INNER_JOIN: ['file1', 'risk_assessments'] on customer_id
    LEFT_JOIN: ['file1', 'loan_applications'] on customer_id
🔍 Parsing file4.sql
⚠️ Failed to parse file4: Invalid expression / Unexpected token. Line 14, Col: 11.
  ON f3.customer_id = f2.customer_id
LEFT JOIN compliance_checks c ON f3.customer_id = c.customer_id
 alert_logs a ON f2.transaction_id = a.transaction_id(+)
WHERE f3.risk_category IN ('HIGH', 'MEDIUM')
OR f2.amo
🔍 Parsing file5.sql
    LEFT_JOIN: ['file1', 'file3'] on customer_id
    LEFT_JOIN: ['file1', 'file4'] on customer_id
    INNER_JOIN: ['file1', 'f2'] on customer_id
    LEFT_JOIN: ['file1', 'investment_portfolios'] on customer_id
    LEFT_JOIN: ['file1', 'credit_ratings'] on custome

Testing code

In [25]:
import os
import json
from typing import Dict, Any
import sqlglot
from sqlglot import exp


class SQLDependencyParser:
    def __init__(self, sql_folder: str):
        self.sql_folder = sql_folder
        self.dependencies: Dict[str, Any] = {}
        self.table_aliases: Dict[str, Dict[str, str]] = {}

    # ---------------------- Utility ----------------------
    def _normalize_name(self, name: str) -> str:
        """Normalize SQL identifiers for consistent dependency mapping."""
        return name.strip().lower() if name else ""

    # ---------------------- Main ----------------------
    def parse_all_sql_files(self) -> Dict[str, Any]:
        """Parse all .sql files in the given folder and build the dependency graph."""
        for file_name in os.listdir(self.sql_folder):
            if file_name.endswith(".sql"):
                file_path = os.path.join(self.sql_folder, file_name)
                print(f"🔍 Parsing {file_name}")
                self._parse_sql_file(file_path)

        # Add missing base tables after all files parsed
        self._add_missing_base_tables()
        return self.dependencies

    def _parse_sql_file(self, file_path: str):
        file_key = self._normalize_name(os.path.splitext(os.path.basename(file_path))[0])
        with open(file_path, "r", encoding="utf-8") as f:
            sql_content = f.read()

        try:
            parsed = sqlglot.parse_one(sql_content, read="oracle")
        except Exception as e:
            print(f"⚠️ Failed to parse {file_key}: {e}")
            return

        self.dependencies[file_key] = {
            "depends": [],
            "left_join": [],
            "right_join": [],
            "inner_join": [],
            "columns": {},
        }

        self._extract_table_aliases(parsed, file_key)
        self._extract_table_references(parsed, file_key)
        self._extract_join_info(parsed, file_key)
        self._extract_implicit_oracle_joins(parsed, file_key)
        self._extract_selected_columns(parsed, file_key)

    # ---------------------- Table & Alias Extraction ----------------------
    def _extract_table_aliases(self, expression: exp.Expression, file_key: str):
        aliases = {}
        for table in expression.find_all(exp.Table):
            alias = self._normalize_name(table.alias_or_name)
            name = self._normalize_name(table.name)
            aliases[alias] = name
        self.table_aliases[file_key] = aliases

    def _extract_table_references(self, expression: exp.Expression, file_key: str):
        tables = {self._normalize_name(t.name) for t in expression.find_all(exp.Table)}
        self.dependencies[file_key]["depends"].extend(sorted(tables))

    # ---------------------- Join Analysis ----------------------
    def _get_join_type_correct(self, join: exp.Join) -> str:
        kind = join.args.get("kind")
        if kind:
            kind = str(kind).upper().strip()

        if not kind or kind in {"", "JOIN"}:
            raw_sql = str(join)
            if "LEFT JOIN" in raw_sql.upper():
                kind = "LEFT"
            elif "RIGHT JOIN" in raw_sql.upper():
                kind = "RIGHT"
            elif "FULL JOIN" in raw_sql.upper() or "OUTER JOIN" in raw_sql.upper():
                kind = "FULL"
            else:
                kind = "INNER"

        # Oracle (+) detection
        on_expr = join.args.get("on")
        if on_expr:
            on_str = on_expr.sql(dialect="oracle").upper()
            if "(+)" in on_str and "=" in on_str:
                eq_expr = list(on_expr.find_all(exp.EQ))
                if eq_expr:
                    eq_str = eq_expr[0].sql(dialect="oracle").upper()
                    if "(+)" in eq_str.split("=")[0]:
                        kind = "RIGHT"
                    elif "(+)" in eq_str.split("=")[1]:
                        kind = "LEFT"

        if "LEFT" in kind:
            return "left_join"
        elif "RIGHT" in kind:
            return "right_join"
        elif "FULL" in kind or "OUTER" in kind:
            return "inner_join"
        else:
            return "inner_join"

    def _extract_column_from_expression(self, expr: exp.Expression) -> str:
        if isinstance(expr, exp.Column):
            return expr.name
        return str(expr)

    def _extract_join_info(self, expression: exp.Expression, file_key: str):
        for join in expression.find_all(exp.Join):
            join_type = self._get_join_type_correct(join)
            on_condition = join.args.get("on")
            if not on_condition:
                continue

            eq_conditions = [c for c in on_condition.find_all(exp.EQ)]
            for eq in eq_conditions:
                left_table = self._normalize_name(getattr(eq.left, "table", None))
                right_table = self._normalize_name(getattr(eq.right, "table", None))
                left_name = self._normalize_name(self.table_aliases[file_key].get(left_table, left_table))
                right_name = self._normalize_name(self.table_aliases[file_key].get(right_table, right_table))
                tables = [left_name, right_name]
                column = self._extract_column_from_expression(eq.left)

                join_data = {"table": tables, "column": column}
                self.dependencies[file_key][join_type].append(join_data)
                print(f"    {join_type.upper()}: {tables} on {column}")

    # ---------------------- Oracle (+) Implicit Joins ----------------------
    def _extract_implicit_oracle_joins(self, expression: exp.Expression, file_key: str):
        for where in expression.find_all(exp.Where):
            for condition in where.find_all(exp.EQ):
                cond_str = condition.sql(dialect="oracle").upper()
                if "(+)" not in cond_str:
                    continue

                left_expr = condition.left
                right_expr = condition.right
                left_table = self._normalize_name(getattr(left_expr, "table", None))
                right_table = self._normalize_name(getattr(right_expr, "table", None))
                left_name = self._normalize_name(self.table_aliases[file_key].get(left_table, left_table))
                right_name = self._normalize_name(self.table_aliases[file_key].get(right_table, right_table))

                if "(+)" in cond_str.split("=")[0]:
                    join_type = "right_join"
                    tables = [left_name, right_name]
                    column = self._extract_column_from_expression(right_expr)
                else:
                    join_type = "left_join"
                    tables = [left_name, right_name]
                    column = self._extract_column_from_expression(left_expr)

                join_data = {"table": tables, "column": column or "unknown"}
                self.dependencies[file_key][join_type].append(join_data)
                print(f"    {join_type.upper()} (Oracle implicit): {tables} on {column}")

    # ---------------------- Column Mapping ----------------------
    def _extract_selected_columns(self, expression: exp.Expression, file_key: str):
        columns_map = {}
        for col in expression.find_all(exp.Column):
            table = self._normalize_name(getattr(col, "table", None))
            column = col.name
            table_name = self._normalize_name(self.table_aliases[file_key].get(table, table))
            if table_name:
                columns_map.setdefault(table_name, set()).add(column)

        self.dependencies[file_key]["columns"] = {
            k: sorted(list(v)) for k, v in columns_map.items()
        }

    # ---------------------- Base Table Placeholder Fix ----------------------
    def _add_missing_base_tables(self):
        """Add placeholder entries for base tables not defined as separate SQL files."""
        all_tables = set()
        for entry in self.dependencies.values():
            all_tables.update(self._normalize_name(t) for t in entry["depends"])

        sql_file_keys = {self._normalize_name(k) for k in self.dependencies.keys()}

        for table in sorted(all_tables):
            if table and table not in sql_file_keys:
                self.dependencies[table] = {
                    "depends": [],
                    "left_join": [],
                    "right_join": [],
                    "inner_join": [],
                    "columns": {},
                }

    # ---------------------- Export ----------------------
    def export_to_json(self, output_path: str):
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(self.dependencies, f, indent=4)
        print(f"✅ Exported dependency graph to {output_path}")


if __name__ == "__main__":
    folder = "sample1"  # Folder containing your SQL files
    parser = SQLDependencyParser(folder)
    result = parser.parse_all_sql_files()
    parser.export_to_json("dependency_graph_test.json")


🔍 Parsing file1.sql
    LEFT_JOIN: ['customers', 'accounts'] on customer_id
🔍 Parsing file2.sql
    INNER_JOIN: ['file1', 'transactions'] on customer_id
    LEFT_JOIN: ['transactions', 'branches'] on branch_id
🔍 Parsing file3.sql
    LEFT_JOIN: ['file1', 'file2'] on customer_id
    INNER_JOIN: ['file1', 'risk_assessments'] on customer_id
    LEFT_JOIN: ['file1', 'loan_applications'] on customer_id
🔍 Parsing file4.sql
    INNER_JOIN: ['file3', 'file2'] on customer_id
    LEFT_JOIN: ['file3', 'compliance_checks'] on customer_id
    LEFT_JOIN: ['file2', 'alert_logs'] on transaction_id
🔍 Parsing file5.sql
    LEFT_JOIN: ['file1', 'file3'] on customer_id
    LEFT_JOIN: ['file1', 'file4'] on customer_id
    INNER_JOIN: ['file1', 'f2'] on customer_id
    LEFT_JOIN: ['file1', 'investment_portfolios'] on customer_id
    RIGHT_JOIN: ['file1', 'credit_ratings'] on customer_id
✅ Exported dependency graph to dependency_graph_test.json
